# BTL02 T0403 — EXP-12 LR Scheduler Ablation (Kaggle GPU)
Restore finished MLflow runs from Kaggle Dataset backup; export artifacts under T0403_Scheduler.

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path
import torch
if not torch.cuda.is_available(): raise RuntimeError('Bật Kaggle GPU Accelerator trước khi chạy T0403.')
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient()
def read_secret(name):
    try:
        value=secrets.get_secret(name); return value.strip() if value else None
    except Exception: return None
OUTPUT=Path('/kaggle/working/BTL02/T0403_Scheduler'); MLRUNS=OUTPUT/'mlruns'; CKPT=OUTPUT/'checkpoints'; LOGS=OUTPUT/'logs'
for folder in (OUTPUT,MLRUNS,CKPT,LOGS): folder.mkdir(parents=True,exist_ok=True)
(MLRUNS/'.trash').mkdir(exist_ok=True)
os.environ.update({'MLFLOW_TRACKING_URI':MLRUNS.resolve().as_uri(),'MLFLOW_ALLOW_FILE_STORE':'true'})
for i in range(torch.cuda.device_count()): print('GPU',i,torch.cuda.get_device_name(i))
BRANCH='exp/BTL02_T0403_ablation_lr_scheduler'; URL='https://github.com/Duy-DS/Deep_Learning_Group.git'
REPO=Path('/kaggle/working/Deep_Learning_Group'); PROJECT=REPO/'BTL02'
print('Output:',OUTPUT)

In [ ]:
token=read_secret('GITHUB_TOKEN')
if token and token.startswith('x-access-token:'): token=token.split(':',1)[1]
env=os.environ.copy(); askpass=None
if token:
    fd,askpass=tempfile.mkstemp(prefix='t0403-askpass-')
    with os.fdopen(fd,'w') as f: f.write('#!/bin/sh\ncase \"$1\" in *Username*) echo x-access-token;; *) echo \"$T0403_GITHUB_TOKEN\";; esac\n')
    os.chmod(askpass,0o700); env.update({'GIT_ASKPASS':askpass,'GIT_TERMINAL_PROMPT':'0','T0403_GITHUB_TOKEN':token})
try:
    if REPO.exists():
        subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True,env=env)
        subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True,env=env)
        subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/'+BRANCH],check=True,env=env)
    else: subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',URL,str(REPO)],check=True,env=env)
finally:
    if askpass and os.path.exists(askpass): os.remove(askpass)
    env.pop('T0403_GITHUB_TOKEN',None); token=None
os.chdir(PROJECT); assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()==BRANCH

In [ ]:
requirements=Path('requirements.txt').read_text(encoding='utf-8').splitlines()
packages=[line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=',1)[0].strip().lower() not in {'torch','torchvision'}]
subprocess.run([sys.executable,'-m','pip','install','-q',*packages],check=True)
from mlflow.tracking import MlflowClient
INPUT=Path('/kaggle/input/datasets'); backups=[]
if INPUT.exists():
    for current,dirs,files in os.walk(INPUT):
        folder=Path(current)
        if folder.name=='mlruns' and any((child/'meta.yaml').is_file() for child in folder.iterdir() if child.is_dir()): backups.append(folder); dirs[:]=[]
def restore_finished_runs(source,destination):
    source,destination=Path(source),Path(destination); client=MlflowClient(tracking_uri=source.resolve().as_uri()); copied=[]
    for experiment in client.search_experiments():
        src_exp=source/str(experiment.experiment_id); dst_exp=destination/str(experiment.experiment_id); dst_exp.mkdir(parents=True,exist_ok=True)
        if (src_exp/'meta.yaml').is_file() and not (dst_exp/'meta.yaml').exists(): shutil.copy2(src_exp/'meta.yaml',dst_exp/'meta.yaml')
        for run in client.search_runs([experiment.experiment_id],filter_string="attributes.status = 'FINISHED'",max_results=10000):
            src_run=src_exp/run.info.run_id; dst_run=dst_exp/run.info.run_id
            if src_run.is_dir() and not dst_run.exists(): shutil.copytree(src_run,dst_run); copied.append(run.info.run_id)
    return copied
restored=[]
for backup in backups: restored.extend(restore_finished_runs(backup,MLRUNS))
(MLRUNS/'.trash').mkdir(exist_ok=True); os.environ['MLFLOW_TRACKING_URI']=MLRUNS.resolve().as_uri()
print('Restored FINISHED runs:',len(restored))

In [ ]:
from src.data.downloader import FER2013Downloader
raw=Path('data/raw/fer2013.csv'); raw.parent.mkdir(parents=True,exist_ok=True)
competition=Path('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge')
competition_csv=competition/'fer2013.csv'; archive=competition/'fer2013.tar.gz'
if not raw.is_file() and competition_csv.is_file(): shutil.copy2(competition_csv,raw)
if not raw.is_file() and not competition_csv.is_file() and competition.exists():
    matches=list(competition.rglob('fer2013.csv'))
    if matches: shutil.copy2(matches[0],raw)
if not raw.is_file() and archive.is_file():
    with tempfile.TemporaryDirectory(prefix='t0403-fer2013-',dir=OUTPUT) as temp:
        candidates=FER2013Downloader._candidates(archive,archive.name,Path(temp))
        if not candidates: raise RuntimeError('Competition archive không chứa CSV FER2013.')
        FER2013Downloader.validate_csv(candidates[0]); shutil.copy2(candidates[0],raw)
if not raw.is_file():
    token=read_secret('KAGGLE_API_TOKEN')
    if not token: raise RuntimeError('Thiếu competition input và Secret KAGGLE_API_TOKEN.')
    os.environ['KAGGLE_API_TOKEN']=token; token=None
    try: subprocess.run([sys.executable,'-m','src.data.downloader'],cwd=PROJECT,check=True)
    finally: os.environ.pop('KAGGLE_API_TOKEN',None)
counts=FER2013Downloader.validate_csv(raw); assert counts=={'Training':28709,'PublicTest':3589,'PrivateTest':3589},counts
outputs=PROJECT/'outputs'; outputs.mkdir(exist_ok=True)
for name,target in (('checkpoints',CKPT),('logs',LOGS)):
    link=outputs/name
    if link.exists() and not link.is_symlink():
        if any(p.name!='.gitkeep' for p in link.iterdir()): raise RuntimeError(f'Không ghi đè output có dữ liệu: {link}')
        for p in link.iterdir(): p.unlink()
        link.rmdir()
    if link.is_symlink() and link.resolve()!=target.resolve(): raise RuntimeError(f'Symlink không đúng: {link}')
    if not link.exists(): link.symlink_to(target,target_is_directory=True)
print('FER2013 splits:',counts)

In [ ]:
import pandas as pd, yaml
from src.train.trainer import Trainer
from src.train.mlflow_logger import MLflowLogger
CONFIG_FILES=['exp12_sched_step_or_plateau.yaml','exp12_sched_cosine.yaml','exp12_sched_warmup_cosine.yaml']
configs=[]
for filename in CONFIG_FILES:
    cfg=yaml.safe_load((Path('experiments/ablation')/filename).read_text(encoding='utf-8'))
    cfg['run_name']=Trainer._format_run_name(exp_id=cfg['exp_id'],architecture=cfg['architecture'],optimizer=cfg['optimizer'],pretrained=cfg['pretrained'],image_size=cfg['image_size'],augmentation_mode=cfg['augmentation_mode'],loss_name=cfg['loss']['name'],scheduler_name=cfg['scheduler'])
    configs.append(cfg)
client=MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI']); exp=client.get_experiment_by_name('BTL02_FER2013_PyTorch'); finished=set()
if exp:
    for run in client.search_runs([exp.experiment_id],max_results=10000):
        if run.info.status=='FINISHED': finished.add(run.data.tags.get('mlflow.runName',''))
assert len(MLflowLogger.CSV_COLUMNS)==24
display(pd.DataFrame([{'config':f,'scheduler':c['scheduler'],'run_name':c['run_name'],'status':'FINISHED' if c['run_name'] in finished else 'PENDING'} for f,c in zip(CONFIG_FILES,configs)]))
CSV=LOGS/'t0403_ablation_scheduler_24col.csv'
def export_results():
    logger=MLflowLogger(); raw=LOGS/'t0403_all_runs_export.csv'; logger.export_summary_csv(str(raw))
    frame=pd.read_csv(raw,encoding='utf-8-sig',keep_default_na=False); assert frame.columns.tolist()==MLflowLogger.CSV_COLUMNS
    names={c['run_name'] for c in configs}; frame=frame[frame['Name'].isin(names)&frame['Status'].astype(str).str.upper().eq('FINISHED')]
    temp=CSV.with_name(CSV.name+'.sync-tmp'); frame.to_csv(temp,index=False,encoding='utf-8-sig'); os.replace(temp,CSV); raw.unlink(missing_ok=True); return frame
failures=[]
for filename,cfg in zip(CONFIG_FILES,configs):
    if cfg['run_name'] in finished: print('[AUTO-SKIP FINISHED]',cfg['run_name']); export_results(); continue
    result=subprocess.run([sys.executable,'-u','-m','src.train.trainer','--config',str(Path('experiments/ablation')/filename)],cwd=PROJECT)
    if result.returncode: failures.append((filename,result.returncode))
    export_results()
final=export_results(); assert final.columns.tolist()==MLflowLogger.CSV_COLUMNS
print('Finished rows:',len(final),'/3; failures:',failures,'CSV:',CSV,'Save Version → Quick Save:',OUTPUT)
display(final[['Name','Run ID','Status','metrics.best_epoch','metrics.train_loss','metrics.train_acc','metrics.best_val_acc','metrics.private_test_acc','metrics.macro_f1']])